In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd


qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [2]:
files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
files

['/home/ulyanov/data/cross/hmi.M_45s.20240319_152445_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.M_45s.20241120_001630_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.M_45s.20250313_180430_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.M_720s.20241120_001200_TAI.3.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.V_45s.20241120_001500_TAI.2.Dopplergram.fits',
 '/home/ulyanov/data/cross/hmi.V_720s.20241120_001200_TAI.3.Dopplergram.fits',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20240319T152009_V202602220014_0443190639.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20241120T001503_V202602220151_0451200501.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20250313T180009_V202602220258_0543130503.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-vlos_20241120T001503_V202602220151_0451200501.fits.gz']

In [3]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [95]:
file_hmi = files[0]
file_fdt = files[-4]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data

did = int(file_fdt.split('_')[-1].split('.')[0])

data_fdt = undistort(data_fdt, header_fdt, xd, yd)
data_hmi = rebin(data_hmi, 6, update_header=header_hmi)

data_hmi = reproject(data_hmi, header_hmi, header_fdt, correct_mu=False, mu_thr=0.1,
                     xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], crota=header_fdt['CROTA'] + 0.25, rsun=ru_sun[dids == did][0])

data_fdt[np.isnan(data_hmi)] = np.nan

In [97]:
xx, yy = [0], [0]

for q in np.arange(2., 30.25, 0.5):
    w = np.sqrt(data_hmi ** 2 + data_fdt ** 2)
    t = np.where(w < q ** 2)
    x, y = data_hmi[t], data_fdt[t]
    w = 1#w[t]

    A = np.array([[np.mean(w * x ** 2), np.mean(w * x * y)],
                  [np.mean(w * x * y), np.mean(w * y ** 2)]])

    vals, vecs = np.linalg.eigh(A)
    u, v = vecs[:, 1]
    u, v = u * q ** 2 * np.sign(u), v * q ** 2 * np.sign(u)

    xx = [-u] + xx + [u]
    yy = [-v] + yy + [v]

print(u / v)


xx = np.array(xx)
yy = np.array(yy)

1.0947021174124567


In [83]:
plt.figure(figsize=(10,10))
plt.scatter(data_hmi, data_fdt, s=0.05)
plt.plot([-u,u],[-v,v], '--', color='black', lw=1.5)
#plt.plot(xx, yy, '--', color='black', lw=1)


plt.xlabel('HMI, G')
plt.ylabel('FDT, G')

#plt.xlim(-40,40)
#plt.ylim(-40,40)

plt.xlim(-400,400)
plt.ylim(-400,400)

plt.grid(True)
plt.tight_layout()

In [87]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi, cmap='seismic', vmin=-200, vmax=200)
#plt.axis('off')
plt.tight_layout()

In [88]:
plt.figure(figsize=(10,10))
plt.imshow(data_fdt, cmap='seismic', vmin=-200, vmax=200)
#plt.axis('off')
plt.tight_layout()

In [17]:
np.savez('cross_calibration.npz', hmi=xx, fdt=yy)

In [11]:
header_fdt['EAR_TDEL'] / 60

1.4438210942356717

In [45]:
view_fdt = View.from_header(header_fdt)
view_hmi = View.from_header(header_hmi)

u = view_fdt.to_carrington(origin='helioprojective')((0,0,1))[0]
v = view_hmi.to_carrington(origin='helioprojective')((0,0,1))[0]

In [48]:
np.arccos(np.sum(np.array(u) * np.array(v))) * 180 / np.pi

np.float64(1.933449871103591)

In [49]:
header_fdt['DSUN_AU']

0.81446896